# 00 · Setup & mental model  (⏱ ~30 min)

**Goal:** by the end you can say what a brain image *is* to a computer, and where nilearn sits in the Python ecosystem.

## The mental model
- A brain scan is a **NumPy array + an affine matrix** (the affine says where each voxel sits in real-world mm). Nilearn reads/writes these through `nibabel`.
- Structural MRI → **3D** array `(x, y, z)`. Functional MRI (fMRI) → **4D** `(x, y, z, time)`.
- Nilearn's job: make the *statistics and machine learning* side of this feel like **scikit-learn**: objects with `.fit()` / `.transform()`, and functions that accept either file paths or image objects.

```
 datasets  ->  image ops (load, resample, smooth, mask)  ->  maskers (4D -> 2D matrix)  ->  scikit-learn / GLM / connectivity  ->  plotting
```
The big conceptual jump of the whole day is the **masker**: it turns `(x,y,z,time)` into `(time, features)` — a table ML understands.

In [1]:
!pip install -q nilearn
import nilearn, numpy as np, pandas as pd, matplotlib.pyplot as plt
print("nilearn", nilearn.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
nilearn 0.14.1


**Data tip:** nilearn downloads datasets to `~/nilearn_data`, which Colab wipes when the runtime resets.
If you don't want to re-download, mount Drive and pass `data_dir=` to the `fetch_*` functions
(`from google.colab import drive; drive.mount('/content/drive')`).

## Fetch one small dataset
The Haxby dataset (one subject watching pictures of faces, houses, cats...) is used all over the nilearn gallery, so it's worth getting familiar with. The cell below is just plumbing.

In [2]:
from nilearn import datasets
haxby = datasets.fetch_haxby(subjects=[2])
print(haxby.keys())      # a sklearn-style 'Bunch': dict with attribute access

[fetch_haxby] Added README.md to /root/nilearn_data

[fetch_haxby] Dataset created in /root/nilearn_data/haxby2001

[fetch_haxby] Downloading data from https://www.nitrc.org/frs/download.php/7868/mask.nii.gz ...

[fetch_haxby]  ...done. (0 seconds, 0 min)

[fetch_haxby] Downloading data from http://data.pymvpa.org/datasets/haxby2001/MD5SUMS ...

...

[fetch_haxby] Downloaded 290807808 of 291168628 bytes (99.9%%, 00 HR 00 MIN 00 SEC remaining)

[fetch_haxby]  ...done. (9 seconds, 0 min)

[fetch_haxby] Extracting data from /root/nilearn_data/haxby2001/3d2c7644e96a21be22fab60f69663a15/subj2-2010.01.14.tar.gz...

[fetch_haxby] .. done.

dict_keys(['anat', 'func', 'session_target', 'mask_vt', 'mask_face', 'mask_house', 'mask_face_little', 'mask_house_little', 'mask', 'description', 't_r'])


## Exercises
**Predict first (write your guess in a comment before running anything):**
1. What do you think `haxby.func` contains: a 4D array, or something else?  
    A: You should get a set of 3D images plus a time.
2. Which axis of the functional image will be time?  
    A: Likely the last axis

**Do:**
- **Ex 1.** Use `nilearn.image.load_img` on the functional file. What type do you get back? (Look at its `.shape`, `.affine`, and `.header.get_zooms()`.)
- **Ex 2.** From the affine alone, can you work out the voxel size? (Hint: the diagonal. Then check with `get_zooms()`.) What is the 4th zoom value?
- **Ex 3.** Use `nilearn.image.mean_img` to collapse time. What shape comes out? Then use `nilearn.image.index_img` to pull out just the 10th volume.
- **Ex 4 (stretch).** Read the docstring of `nilearn.image.load_img` with `?`. Why does nilearn accept *both* filenames and image objects everywhere?

In [5]:
haxby['description']

'.. _haxby_dataset:\n\nHaxby dataset\n=============\n\nAccess\n------\nSee :func:`nilearn.datasets.fetch_haxby`.\n\nNotes\n-----\nResults from a classical :term:`fMRI` study that investigated the differences between\nthe neural correlates of face versus object processing in the ventral visual\nstream. Face and object stimuli showed widely distributed and overlapping\nresponse patterns.\n\nSee :footcite:t:`Haxby2001`.\n\nContent\n-------\nThe "simple" dataset includes:\n    :\'func\': Nifti images with bold data\n    :\'session_target\': Text file containing run data\n    :\'mask\': Nifti images with employed mask\n    :\'session\': Text file with condition labels\n\nThe full dataset additionally includes\n    :\'anat\': Nifti images with anatomical image\n    :\'func\': Nifti images with bold data\n    :\'mask_vt\': Nifti images with mask for ventral visual/temporal cortex\n    :\'mask_face\': Nifti images with face-responsive brain regions\n    :\'mask_house\': Nifti images with house

In [6]:
# Ex 1
func_image = nilearn.image.load_img(haxby['func'])
type(func_image)

#You get a set of images with an affine mask to translate the images back into real world dimensions

nibabel.nifti1.Nifti1Image

In [10]:
# Ex 2
print(func_image.affine)
print(func_image.shape)
print(func_image.header.get_zooms())

#One voxel is 3.5x3.75x3.75 mm
#The fourth zoom is time in seconds, and the last column is a neat way of sneaking in translation into the matrix transform

[[  -3.5      0.       0.      68.25 ]
 [   0.       3.75     0.    -118.125]
 [   0.       0.       3.75  -118.125]
 [   0.       0.       0.       1.   ]]
(40, 64, 64, 1452)
(np.float32(3.5), np.float32(3.75), np.float32(3.75), np.float32(2.5))


In [11]:
# Ex 3
mean_img = nilearn.image.mean_img(func_image)
print(mean_img.shape)
print(nilearn.image.index_img(func_image, 10).shape)

(40, 64, 64)
(40, 64, 64)


In [12]:
# Ex 4
?nilearn.image.load_img

#My guess is datasets can get really large, thus allowing for calling from a file rather than memory would be convenient

## Check your understanding (answer in your own words in a markdown cell)
- What does the affine do, and why would two scans of the same brain have different ones?
- Why is the 4th "voxel size" of an fMRI scan not really a size?
- Why is keeping everything as one `Nifti1Image` object (not a bare NumPy array) useful?

## Where this lives in the gallery
The `examples/` folder in the nilearn repo is numbered by theme: `00_tutorials`, `01_plotting`, `02_decoding`, `03_connectivity`, `04_glm_first_level`, `05_glm_second_level`, `06_manipulating_images`, `07_advanced`. Today's notebooks follow roughly that order.

**Read after you finish this notebook** (your own attempt first!):
- `00_tutorials/plot_nilearn_101.py`: the official "what is nilearn" walkthrough
- `00_tutorials/plot_3d_and_4d_niimg.py`: 3D vs 4D images, exactly this notebook's topic
- `00_tutorials/plot_python_101.py`: only if NumPy/Python basics feel shaky

What does the affine do, and why would two scans of the same brain have different ones?  

A: An affine translates and transforms data from an array to actual positions in real life. An affine could differ for a number of reasons, from different scanning sequences/settings to a person sitting in a different position and preprocessing accounting for that.  

Why is the 4th "voxel size" of an fMRI scan not really a size?  

A: It's a marker of time

Why is keeping everything as one Nifti1Image object (not a bare NumPy array) useful?

A: You keep the affine (which defines transformation to real world data) alongside the data it corresponds with. Keeping it all bundled together prevents mishandling and confusion.